# Level 8: Quantum Phase Estimation & Shor's Algorithm
## Q# Edition

In this notebook, you will:

1. Implement **Quantum Phase Estimation (QPE)** in Q#
2. Use Q# standard library phase estimation tools
3. Understand how QPE enables Shor's algorithm
4. Explore **resource estimation** for real-world factoring

In [ ]:
import qsharp
from qsharp_widgets import Circuit
print("Q# ready!")

---
## 1. QPE in Q#

**Quantum Phase Estimation**: Given $U|\psi\rangle = e^{2\pi i \theta}|\psi\rangle$, find $\theta$.

Steps:
1. Counting qubits in superposition
2. Controlled-$U^{2^k}$ operations
3. Inverse QFT on counting register
4. Measure to get $\theta$

In [ ]:
%%qsharp

open Microsoft.Quantum.Math;
open Microsoft.Quantum.Diagnostics;

// Inverse QFT for QPE
operation InverseQFT(qs : Qubit[]) : Unit is Adj {
    let n = Length(qs);

    // Swap qubits
    for i in 0..n / 2 - 1 {
        SWAP(qs[i], qs[n - 1 - i]);
    }

    // Reverse QFT operations
    for i in n - 1..-1..0 {
        for j in n - 1..-1..i + 1 {
            let k = j - i + 1;
            let angle = -2.0 * PI() / IntAsDouble(1 <<< k);
            Controlled R1([qs[j]], (angle, qs[i]));
        }
        H(qs[i]);
    }
}

In [ ]:
%%qsharp

// QPE for the T gate: T|1> = e^(i*pi/4)|1> → theta = 1/8
operation QPE_T_Gate(nCounting : Int) : Int {
    use counting = Qubit[nCounting];
    use eigenstate = Qubit();

    // Prepare eigenstate |1>
    X(eigenstate);

    // Hadamard on counting qubits
    for q in counting { H(q); }

    // Controlled-T^(2^k)
    for k in 0..nCounting - 1 {
        let angle = PI() / 4.0 * IntAsDouble(1 <<< k);
        Controlled R1([counting[k]], (angle, eigenstate));
    }

    // Inverse QFT
    InverseQFT(counting);

    // Measure and convert to integer
    mutable result = 0;
    for i in 0..nCounting - 1 {
        if M(counting[i]) == One {
            set result = result + (1 <<< i);
        }
    }

    Reset(eigenstate);
    ResetAll(counting);

    return result;
}

In [ ]:
# Run QPE for T gate with 3 counting qubits
print("QPE for T gate (3 counting qubits):")
for i in range(10):
    measured = qsharp.eval("QPE_T_Gate(3)")
    theta = measured / 8
    print(f"  Run {i+1}: measured = {measured}, θ = {measured}/8 = {theta:.4f}")

print(f"\nExpected: θ = 1/8 = 0.125 ✓")

In [ ]:
# Visualize the QPE circuit
Circuit(qsharp.circuit("QPE_T_Gate(3)"))

---
## 2. QPE with Arbitrary Phase

In [ ]:
%%qsharp

// QPE for arbitrary phase theta
operation QPE_Arbitrary(theta : Double, nCounting : Int) : Int {
    use counting = Qubit[nCounting];
    use eigenstate = Qubit();

    X(eigenstate);
    for q in counting { H(q); }

    for k in 0..nCounting - 1 {
        let angle = 2.0 * PI() * theta * IntAsDouble(1 <<< k);
        Controlled R1([counting[k]], (angle, eigenstate));
    }

    InverseQFT(counting);

    mutable result = 0;
    for i in 0..nCounting - 1 {
        if M(counting[i]) == One {
            set result = result + (1 <<< i);
        }
    }

    Reset(eigenstate);
    ResetAll(counting);
    return result;
}

// Statistics: run many times and track results
operation QPE_Statistics(theta : Double, nCounting : Int, shots : Int) : Int[] {
    let nStates = 1 <<< nCounting;
    mutable counts = [0, size = nStates];

    for _ in 0..shots - 1 {
        let result = QPE_Arbitrary(theta, nCounting);
        set counts w/= result <- counts[result] + 1;
    }

    return counts;
}

In [ ]:
import matplotlib.pyplot as plt

# QPE for θ = 1/3 with 5 counting qubits
theta = 1/3
n_count = 5
counts = qsharp.eval(f"QPE_Statistics({theta}, {n_count}, 5000)")

n_states = 2**n_count
labels = [f"{i}/{n_states}" for i in range(n_states)]

# Show only non-zero counts
plt.figure(figsize=(14, 5))
plt.bar(range(n_states), counts, color='steelblue')
plt.xlabel('Measured value')
plt.ylabel('Count')
plt.title(f'QPE for θ = 1/3 ({n_count} counting qubits, 5000 shots)')
plt.xticks(range(n_states), labels, rotation=90, fontsize=7)
plt.axvline(x=theta * n_states, color='r', linestyle='--', label=f'True θ × {n_states} = {theta * n_states:.1f}')
plt.legend()
plt.tight_layout()
plt.show()

# Top results
sorted_idx = sorted(range(n_states), key=lambda i: counts[i], reverse=True)
print("Top 3 results:")
for idx in sorted_idx[:3]:
    est_theta = idx / n_states
    print(f"  {idx}/{n_states} = {est_theta:.4f} (error: {abs(est_theta - theta):.4f}, count: {counts[idx]})")

---
## 3. The Road to Shor's Algorithm

### Period Finding

For factoring $N = 15$ with $a = 7$:
- $7^1 \mod 15 = 7$
- $7^2 \mod 15 = 4$  
- $7^3 \mod 15 = 13$
- $7^4 \mod 15 = 1$ ← Period $r = 4$!

Then: $\gcd(7^2 - 1, 15) = \gcd(48, 15) = 3$ and $\gcd(7^2 + 1, 15) = \gcd(50, 15) = 5$

So $15 = 3 \times 5$ ✓

In [ ]:
%%qsharp

// Classical verification of the period in Q#
function FindPeriodClassical(a : Int, N : Int) : Int {
    mutable power = a % N;
    mutable r = 1;

    while power != 1 {
        set power = (power * a) % N;
        set r += 1;
    }

    return r;
}

In [ ]:
period = qsharp.eval("FindPeriodClassical(7, 15)")
print(f"Period of 7^x mod 15 = {period}")
print(f"\nFactors: gcd(7^{period//2} - 1, 15) = gcd(48, 15) = 3")
print(f"         gcd(7^{period//2} + 1, 15) = gcd(50, 15) = 5")
print(f"\n15 = 3 × 5 ✓")

---
## 4. Resource Estimation Preview

How many qubits would we need to factor RSA-2048?

| Factor | Estimate |
|--------|----------|
| Logical qubits | ~4,000 |
| T gates | ~10 billion |
| Physical qubits (with error correction) | ~20 million |
| Runtime | ~8 hours |

Q# and Azure Quantum provide resource estimation tools to get precise requirements for any algorithm.

---
## 5. Key Takeaways

| Concept | Description |
|---------|-------------|
| **QPE** | Find eigenphases with exponential precision |
| **Period finding** | QPE reveals the period of modular exponentiation |
| **Shor's** | Factor integers in polynomial time |
| **Adjoint** | Q#'s `Adj` functor makes inverse operations easy |

---
## 6. Challenges

1. **QPE for S gate**: $S|1\rangle = i|1\rangle$, so $\theta = 1/4$. Verify with 4 counting qubits.
2. **Higher precision**: Run QPE for $\theta = 1/5$ with 3, 5, and 8 bits. Plot the error vs precision.
3. **Period exploration**: Use `FindPeriodClassical` for different values of $a$ and $N=15$.

In [ ]:
%%qsharp

// Your challenge code here!

---
**Next up: [Level 9 — NISQ and Hybrid Algorithms](../Level_09_NISQ_Hybrid/Level_09_QSharp.ipynb)**